# Laboratorio 1.4 · Procesamiento en un solo nodo

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** tres años de ventas de la tienda, **10 millones de líneas de pedido**

Una sola pregunta de negocio, cinco formas de responderla:

> **Ingreso neto mensual (en MXN) y unidades por categoría y ciudad, solo 2025 y solo canal web.**

| Parte | Qué haces | Lo que vas a descubrir |
|---|---|---|
| 1 | Generar el historial | Cuánto pesa lo mismo en CSV y en Parquet |
| 2 | La misma pregunta en pandas, Polars y DuckDB | Tiempo y memoria pico de cada motor |
| 3 | El plan de Polars | Cómo el optimizador lee solo lo necesario |
| 4 | Parquet por dentro | Row groups, estadísticas min/max y particiones |
| 5 | Más grande que la memoria | Por partes, streaming y fuera de memoria |
| 6 | Tipos de datos | La misma columna puede ocupar 10 veces más |
| 7 | Arrow | Pasar datos entre motores sin copiar |

## Preparación

In [ ]:
%%capture
!pip -q install -U "polars>=1.10" "duckdb>=1.1" pyarrow

In [ ]:
# @title Instalar el paquete del kit (el mismo código que el kit local)
import base64, io, zipfile
zipfile.ZipFile(io.BytesIO(base64.b64decode("UEsDBBQAAAAIAPazRV326cQ9ZAAAAGwAAAAZAAAAcHJvY2VzYW1pZW50by9fX2luaXRfXy5weQXBwQkCQQwF0LtVfHJSkAXBk0fZAraFuBMwGPNlMnuwLkvYxnxPRJbO1UrfbjkIS2yJYhDJxhs+mk3rjIWhvfDFvK2v+Y7ioxss8PQa7K6BZgjFcMumOJaV77/EZbqeJhE5/AFQSwMEFAAAAAgAArRFXceiEs12BwAARBIAACIAAABwcm9jZXNhbWllbnRvL2dlbmVyYXJfaGlzdG9yaWFsLnB5hVjbbuNGEn3nVxQ6L9SEpnWxnYkALqAZey7BOOudyyKAIRBlskT1LtnN7W7K4wT5mDzmYR6CfIJ/bFHNiyRbnhFgieyue1edrrIQ4o20ThuJJeQEG1IOLT+VCE6SyhEQyGZY4hwqWZZaUbt//0VRS1pTLnMdgaHa6LzJ5E1JNg6CT1ZDqTMs5wD1nVtrBUcV1EZnZLGSpJyOC1Jk0KTrwYqjo5Us0cJk3H7g6MhiKXOEHJ22W8oAdj5hphU8h9cv2KD3i0vQUJHSNoLG4iBz1oocBRc2M/KG5gFAKdmPOLMbeOrTKLC61PDyw7+hMKhyYn0VeweknKECzf0X9HTSOqoQSiow16Ot/BrN/xpyB+VTCZW0lYZMK0dK5ix20HrVcg6iUm1yUpjrQzL7Pai1gRVla4SwRoOwIcN6LJZOc4yMvoXC6Ka2WyPTGo2TmdQs4RiV1MlisVgcV2STy8vjZzsKdymBrJOlhjdyQwFAVvLR0uByBG1aOD0sBUKIQFa1Ng7QFDUaS/27XTdOlv2bkxUFK6MrqNGtS3kD3cYVunXQU6mmqu8ALai6X6p1icbyWl0GwYeLy7fv3i0ggR+f/3B2ehK8fPvpfHF+8QESuBYvzy9/ERGI1w3mWOJ/0CC/XvJpGEN3/HLV0E3pl//VkLn/06HRnuj+TyNzv/FOVv73hS60u/+DHz+gchILT/mysZkWy+Dl4uPF63++f7tolV+UlDlz/7eSmed+ows0/PBTUzTkyPLze137zXNi59q1F1SW9Gun+cZov3iJNtMO/fNPaPL7L8rr1plU6JX/vHjXuX1LN7yHdc0/bbWLZRAEOa245o28kWYvJ8J8FUHOx610BJkum0qhHc19KQohrp7IinnP1CbVdDw99Uk1mR6jydZyM+QyJwYLc+ZuPhR4vopvjXSUdkThYII3zkmt0pu7ZLDHM9LnjGoHH+9qujBGmzkAfAdXXVooJ4tGz7mC15hRpS0XH9R3aIy+HTT3ydQuxzk6tOQ4q3K7Nc925nXbYb6KnU49SzjaCdhKmwpdInpfd+yXqhjs319OVyVutEnEWm5IjLrj6XAzbKExAo9vyWScjsf+LwKV9oWYnA5LQyUm03bJEoM6Jl2BdEfZAW7iq6zT0UZVrrrNmD5L62w42h5TW7mxqZwh2uPqWKr/5tIwHJFyNvloGmq3jSogAVXHjK26inNaYVO61Kgi7AzstCvOLkg8LMQ1mVWa6YarNBwFnqL3GRKoy/gcHb4yWFH422Cl6EhSmYs5K0WDqqBwshsy+B4mEeTurqZE1bFU7uxkFO3IkE2OzG9UEWdrLTMKe0jZlRNBnVzHs3EE8eSEv2YRxOPn/HXGX6e8xrvjH/iLScaz5a4mS0XFN+W+rmuhGtp4WFkZyhpWxi8bWYvlYwNY12zqVffCf28jOiTEVwLW0xyM2FbAt0KGjgq+uB9EbUDDPWm7nLWhTOq0lNZha4DRjcpDllLqQnFVleFpfBrBOP7xgRiYbh32v9/BK74WLV+xM8D7v7rSr+7/sH3/QwqU3kiqbgwd5TJrnyB80ZCCV1LBHawkWddjTS6RAxjWJUMEpW18urdwOp7OIg7OZBTBzuJpBJNpBDNeFpNcREBYkGkLozsesrotjds1GQpZUZy7uNLKrcMRw4y/+8IR/COBySSCaTyOYBKPB8NSmX+GZDfkJSkvaNTBBueIV3Ts9cW2qcJOv6WiUbnPDhYglaOCjA3HETw/S08YQbyI/WR6SH0oUfbYtufLadgTXu+f/HLH2+tB1RFMlu3Btj3MV/LYExxI4rY3/Eb6tk1uyx1u2Vs34PgYZixhFKNlEeEhEb4dY3Z/ijvOdKe07JkFZwhD3NnJtV2KEXw/HMRAwts5lQ47mtEh5oZ3oieg7/ER7aNf61r0FA70b3slrpzc4uKu8LNe3n6AZtMDhd4o6dBIvVvreznyzItvlOQLNfQ1P4m3FsOe0Jysx8eHCDrmQtn78vgx8c+T0+WBAFRaUY4Pkfjyl58Zej99OBfLnYq6jp97dD9d7qOgwvIhAvqWbI+VOWf77D1+dfNE23FkdtNdtHDcJbgfZcToMW3fPD2i37YjRt+mfihIrfyVktO2lxh1uHnxeEx5PG7MgYuJsTG//2KdzNBCJdVxhZ957Mgwx+3sAVaaDSnwEwqPJmjaKW/XequNC7viGX3Dl0ez0TecOtzl9mGTbp22TZkNfeNal3Gmy60xuYvvCA03edzOHtjugHonBR5/Hjmxa4yI4Fqwcs6xiqxYjvYanScD8nAMEw8u/CcZH01rHach1xgFvwmfp2Le56vosYnT2hfrge4Mjrrmzd+Ce8EQmd2k1c3A/URCx9ahC/nHnyMcw/TZs+n4obDO5q8LHBx7SujvQRDIFaSpworSFJIERJpWKFWairbjrSEZZtd4YYqG27QrfjM8n2RG1tzBJ+K1b9V5+F5/418t/RHFmOcpdiJD0f33QkTgbyepHM8UvkXe6fif4m2dF1sW8eD/KJ1Wf/XG3h3mt2GfL1K5sB83MO4HDoxbeBwF/wdQSwMEFAAAAAgA6LNFXT78vHy0CQAA6BwAABgAAABwcm9jZXNhbWllbnRvL21vdG9yZXMucHmlWUtv5DYSvutXFBRgLQUdxTPZB2BAB8+4J5mFX3HbmywMo1EtVbeZUKSGpOzpMfxjctxDTnsIsMf4jy0okmqpH3Z21sBgWmSxVA/WVw/FcXyMUDFdIdSKFo0wCCWBoIUsmARFuiFuEAopoGTaMGGkhkoaqUhbykaAllyCkKXMoujcMzkAJhaKtARBRkJFQjfIISEBJz+eprCERrASS9JQSwUFGlpI9fQrwhIK1pRYjhzf1/uv/wJL97tAgRzuaZZFl7JEDRw1zBtRMClIg6KCzUgARyhQ1dSqwuGWaSMVQw5LKOmuIX5Hwsp9hAbfKazI6lGjsBytnsRbi8iI9IeGKiuSVCWJEdSoEGpZkoJCVrWSM1TwoSEwspQaeN80KIA+YmGwImHICuXMrEjXDWmDWRTHcTRXsoIazS1nM2BVLZWBczS3UeQfyqb4uZyFJy8maqjLbk1yVG6NR9Hl+/Oz6dvDkzfvz6ZXkyPI4dXfsv3o7dnx1cnp4QRyuI4r0vEIYm91hu1Da3X7K3jG/vZenFYfRXwTnY7fjieHF+89mzkVt+4sZ1bJKWvP10qWTWGkfyxQGOZZ19ZHctoIZlAxaZdK0kVDwrQPlRRUoj+FPL6JoqikOUznTCBPynl6EAEAlHPIoZxfB61uskLWyyT1m07DG0fjfmeozbKmJGbC/PXP8Yqy0zaQrxZ2nhlYxR8brHUn51xiezZTshFl8nrFZGX9wKK3Ehhoo3onnIs6cv+4QavINEpAOc+0VGZ6h7whnTzr9ps0U6TJTJko6WNSKlnnl6qhNIqiL+Crtb9wDdfXna/c5rSQQjfcYMKZINQjKL3zwv3QkENdZoqwnNaoPjRkkhK+Xl0gnfllb3p/zXaeC/trx9z7Ifc/rr1A4f7eZKXJloQK8ryFmxT+BB2Nv4d2L76nWZzebOOZVaQWlHSCX19vRkHn25sRSJEPCFLPIGhwfb0WU8FN/mxv0+k4x8JItdIxRNJNVmGdPMQnP57GB/Aq2x9BfDU5ig9gDScet9oqQ63ZQiQV6XyL1SopzO2oPbjjrxcVec+kDhBu4MtO3nVksHvJK/iqI1jhxE0KX3qF/X1fybtQsqlnyxcu+whQu5uev0OuKc1wsUhC4OfJALN0U8XpaKBJMoj2QDMIPo9YKvUIFoJC3yU+OfloiOP4WEL19ItNPtXTfwS4FFarp18Kwwo8AE6kbFZ6O/lHm3e4TakIFVVWsTat6bp5+peGOeNGocpsZmlRA/I2m3TvHIi4HqghqKyQbUB5oxb6Lm6Tn6ZpiYZ03t2C1Ib1UMUQkhtqntikCsXTv0u2kCOoSUngtCRRSjh3hyDxqZ63uZg3lUANggrSqBjq9P/UbAAXXrsAFiP/Qp2v0lxQbwsEnrusuwMC280tEDgCjp+W3iatX3OoeaYLFEE4YPOWCIhrspt9yYcAOupDouW2Ez9Hve0dMOlt52UdBHVm7xWpJKl5VkieeO+nATmTdICdgcphZ7rCznTI9SfJxAozM02cCpPsRs50C3Bu4djpFxhuhdJ0B5J2vO6ZuZ36K9EC4BblWwBM0ucgsI8bPcM4eLFIFhbXETANCBgIVgi4ZsjhX8vx/pZE5wefC5wjLPqnmbH7aykgzaS5JXXPNCWvsv11b7XQOp0tk+egde3MAFY3tc90UyVr2Bqo+gjrCdMOaVx4ES5IbeKMj0wSUMlSAhMVlQyNPIACS4QatQRNQD9R0RgE5FBJW6RLiwCV9DjmAapquGG3jMu2CZkpgkOl5P0fxCGXBDbgYC2ot8JR2qFFyFFGelxLVobgFjccd0u6ZowXQHJTqj4OvSRVKA97DtkiwqY/SjYnxUp5ALVilbUxqgptfrPajIBLkLVhFfuEofGzzhEFaQtidj8kBNm5YWjwnWaxV6vFhK3WnGqjCCsmFi0Hr4FRy4PuVvv32O2OF4kFE5TH3WkfBfSxoNrA5bKmsVJSHQDAF3BHSrt+FYUhxUIj7az07Iu6F2yzfbe5vcbwTqiVtKazKdcXFLb9rlHZHLJ6QXoAwpuZGQRDgpRtwEPxYX3pqo//LRBWBt55+7YXH0Ytp/0CpDXA2l3c8OmWrH3UFD8fvdnM2pPvjyG3doom4+Px20twyM6zFuxTOJxARRYTsg71RlBkflQRvDa5Okl4FvCtPRXQbweNrYCzNeTvKt+sQ3ybC94eTsbww3fjU+CZQ3TIYe9qcrQHl3b1wRSPMD6ejOEVjE+P2rf3QDR6d3F2Ag/Oqo/Ao7+fvT+FAQztPZSPX28UD3sp1HA1eX/6LSS91JvuYrBeXuylUITzq2ybRj98N74YQ1s+dGZ2RQQcnh5BayPkVsd7mu1F316cXZ3Dm3/C4fExnF0cjS/8QztGcaHgZiW7K1Dv/XDnfBSE2rNkirp5jR0ETb4/PgDNhB0nLdAGALa1Fr58+TNbCEvNPibb4sDJmekPPJl8f5zNparQJKbI1zLyCMrcXvHWZ/k83jT2EJ730jhNs3K+gjRvkULfTTmrmMFSBhlHQYs8/mZ//+RNvGEnl++6xqMdmvGnXytm2pFZ6D9QGTZnBUPuTDfDn6S1m4UQToC8QPEJR9BotDPEQv5xs9mX5l6JrJBCtNgdtjL6aFM4JfN4Mr508iydnvbaPHgBH/fiXUcMVfXU+V2qZXuofPx66q1mqnov3uY/y+jznGfRre+4Qt8NnbZ72NID6q1NR+h5bhvx86DRHNm2sDWsmrYM8lfT/f19+2/lcv+S0I9tJIeueFKGbPXkOk2boAlwoWiBI1jaWqq1kF0tZDVjAkU7Fq1R2QtC+oV88TmzoeeHLZ85Odo9gXFyBn3sINQtzYOtgAl4vpVuNBWS9xvNHe31CJw72SfK15yYrooS99bc/X+duP+enW55kh3DrQ2e6+OtlwdYzzRX3ajKC/GZk6pNIXuzqk0TvDyqWmvVOgutZlV+beeoyu8/M6kaXJ0M65pE6ZzxuZOr6+vBsGowhb7xbVM3JKtLC6IFmqQTIv3skVnLe9vIK1MksKIkjFMeVhIe9L4uPLZ4d3J2eXYxtt8THhwUeSD6/TcLQPbAanIWj9ZJfOrukXXTHEfqy97ff1u1gS1xr4HcJA0dSo/SVpmB0GfH4euHpUc8ih6jd1fji8Pp0Xh6Mj45u3h/uEPHrqp+SdkhIPepO8jf1KWruDtzbmkXNjULGf+b/X04edNTsF9JtFq6OmOGmjbrrVMJt1gQCCzRVgTqjtw3NOsK1X4T8zWE/Ygmi6ZGAedLcysFLNsZIGczRfa7oMt/+ulX2xGiXm/86jLrPud1Fy98HUqj/wJQSwMEFAAAAAgA1LRFXXCOzhS4BgAA7Q8AABYAAABwcm9jZXNhbWllbnRvL21lZGlyLnB5lVfNbhw3Er73U1S4l+5sDyNrkT0M3AsItgwYkCPBUZDDZNCoadaMqDR/QrJlzRp+pJz2EfxiC7J/R7KC3TlIbJL11Q+rviIZYx9IyEZ+/Y+G+84HXEODAkGZYBw0xjkC0tBpsM405A2gRRcILDoERUI68B0EScoaOMaxImWcRLh5/+YavNRZY3RAJTU6wBYc+WAgb3Ha+EdHSwWt3JFD0AZ8RHUEnkDQQ0ftA0UEL30ghWAsOQzywRQ8u5rhPIGSIop4QYD7jhyWYHae3ANqYYDaSVfy1ZrGeFCylZ4OnRbGQ77vdCONxuj8r1IL88mXmcLm+mc4wpXU3WPBs+yXaK5psF0D2GO4MxpWagBHJUkHw/sYrVYNOksBQWAwvr6TPkRr24wxlklljQuA7mDReRq/773R49h3ux7XTzPHaRhI2b1sJ8EgFWV7ZxRYDHet3MGwcIPhLht3WdQCPaAHK6Y53wXZZr0wT1lAfpR+98vlx4v67WX94fLD9cf3FyV8uL69/nj5c5bdXF38dPv+6uoCKnALl6ILZTS1XBq1DNAIPujKpJaNNFCl/dyS29eN6XQglxeZI9+1AUVcP1DAEFw+CCYtHN3hYfNqW+TTx/m2yKZz/SYqrKBXOsPzYGqL7o+Owoz0j20JUgt6rN5h66nIrJM65NFFLjplff6ZjZrYGpzptMjHiRLOiy9FkY47ywTt++IZE62EIT+KdQYAwBh7k4pv8I6/Hjb+Kx83PqlL3dGD4fB2LJS+JMupLKxsYoFSqsDBy2hM1PZJhrspifhtFHTojm+loyYYd8yLmCVB2d64+PPYSoFQpZTKg7IF/ABshh6ixyYBh/Lf4/a6jorquuCOvGkfKC+iAOkw/AOAv/UUE8mB2jn9DZBuOtLBpXmLUQtNWixUi1LhN8aSzjfxCOmRmi7grqUS2KphJUw5W8J0Cj64PJk4Rnm2sOhXe8eLbTnp/ObPB2G6UC1teX9zGSEEOfd8PtBjqG5dRyU0n0QVo1XMXvWul/0hVkOV8pseIbfcSlGUcDYJfLqTLYHl1rRtXoD08JPRtE5hHZhQGQ8LEnbkpSAdIm+2cCfvDaT6dPHgHe7wHk8cDu64fhaBwTyFj3kclqPhPGk51lLvTV5w531xIkuPDdmRfPilc8Y9B985wt9PbYi17Fsim5/xs/PiSXLWPogSKKLFYYwbb4xSnZYNBsrn/XIPljsKndONEQTfVXB2akC/CCcFHkNaAoue1mo3TySNbD1r5j44afOCe9vK0EpNPi82q1fbzfrVP8+2X8okOelTqSMbDVWiT94aFD6ffXoRrngGsZms20ZiToyUDukHOP/++/OzWWJwcBQswQruCMVMg33eD+TVGBXL0411Uo5EVfVBcGQpJCTy1auZ1G5x1+IoHeQDwt/hgZzcywb7W4igVPPBRL5uY2MempBAHSteSa/MNzhs3FZNI+Om/pTyBz2VUMcNiXlZnGAz8aZNe9mij+bvyZFuJJYgDx22CXizLYcz/vyl320caKN2jiYKAaknzpaBlM+L9eJY7gcbTw48wtRR0KE+UL6MXXGahqqEZQt8oYWciMj9pHaggRiYXG3mVN7GFW1Cv4pawOnq6xFhOVs8r9DZPbUsrdnihZr1UyPnkE909UzBYk81w54i9ae16Y8lZv3OmDZvSeeTQAFVBf3MiFf8NZv3vxiZGWWzYYo8K4FFNjlEFk0fshMo4qjTUqAgz7bbqHBW9v9IFhwjhfd//2cjF1YyqQ+xh9XqUbMtrJZ2nK4VHHc+L+A1nPGzHweVs85UGhytJS3yzywlOVtP+b9kxm+ky0vtkg0NaGbQRcYu6WtMXmrjBf/5+ioV+GLmZY2RQeopQmw9Jg0/UMh7f4oFjw/K0uowWXzp4xISnVWRLN9iwHcOFeUpUAX3FOp0XcyHWPUict9LbQaoLZde43jCc86P0Ok/F87YvDFtp7SvRiMymIk7bcuyTO6hrjUqquuYdKyuFUpd16xHjvej8ZnBL9yhU6TDTfxyuSDfOGmDNLpib3qKntk3vdG8aQ1oIwx4s+tvYNNLZrDHchSixgE6Z9PLh5UgaI9dGyr25BH0omh6vK0ErYYsYSVg0xsYRakOroscfketrdjQVoB8cLGwZHzaxImkLrUVHd98O9Lx8jxi9srTMfIUmGiCH6qtv+NP7Q751PCevobiySJPFteC6vFilfJ16ENFfFfE7q0PY2X1+Ow3PddBIvbFpV0sHqyKdGqLyRnTdBY13PQvzyO06OPr2ZH7+md03TjwX/+Mh4aeFdl/AVBLAwQUAAAACABItEVdq/ZCueUEAAAeCwAAIAAAAHByb2Nlc2FtaWVudG8vcGFycXVldF9pbnRlcm5vLnB5jVbNbttGEL7rKQYLpCIFho4VtAchTOHGDhqg/oF/UBSGQYy5I2kTcpfeXTpyBT1CH6IP0EOvvRSoX6zYXZKikjqpDoaXO/xm5pv5ZsgYOxYaNXCSVivgBI2EM9R3DdkZaPURFlo1tUmAjEX++IexokADlZB7Fa7gAYrm8XdpFRiCuiFOYLC0qAFLmIvSatQpY2wkqlppC1ZUNJprVQFHS+4E7U13Drc12mUpbrvLM7TLUYfBm+IDv+1ONUqOBtBAzftnD6i1+pjWIRF/eTcajTjNXU55yCnSjcUEClU2lcSMzalYIotnIwAAxtiVRJcCQq30lgoolATTgHz8q6JAmTMy8ABU9rxwghI7aIgWDWqOXBkg6cxqQcCpBNTFUtyr2FPk3FZkETKo79K2CG9FST7QOHV3HC16Q8FXkHnz1BRLqjC1KvdZ5+EcxemCbD4XVPJcSE6rqA0n9gAh6Ayub8JRaRAgJGiUC4o8sGyqfEtXS4z76UXnu7+ORNxfk7GQgV6kwWEk+CpOjUUrXPeY3s6HkGJdk+TRmvVYbAYiAeav2cwBtZGYBFglJJs5F2klpDvjqjvjKumxd3+sus0LVdVaVIIrh6kaySO9SK2yWOa3D5ZyI34l2IPpZDJ9kcB+vAkZabKNllDz9BAtvtVYUeRDiz/rqFxSQQa1UF1zcTKcElii+WKr/fP3myAjM9AcnJ0enj/+dnDiWs6SJN0WjSRcD4Hj7+GoBE0myNALEIyQUBLpUpm+u3zdnhJAyNaqAqWv3nWkF9ee3ht4nYVMYvgGwmMh2Q28av3fDIkqSUYeJU78/3rREWUFVbXK/VhQ0bwhaelTijTVZEUhlCSTvdwydOnfDQPKCUuaxmUZRgwPGvWsen0eNsWHwx8gqui9csMN7lELNFAorQVHsxWcuSshgzlj7OLop6M3l/Dm9OrkMprEcHABpZCEJoHz06uTw+ji6jgqUFrBkcMEak2FUHkjhXUVjxOY+pfuSbYa3f7enp8ewzpkvIGffzw6P2qjfZ3B5bvjo4vLg+MzGK89GbNnvzx/Vj1/xjdjODg5bC1f7Rh6ugaG2wHyXmlHo+eHK8jgRElK/N9e6flW6UPCBxIX0j2DzE/stCY9zwvVSEs62gp96CVM5ZRWVDSWInNXxumcbLFUkgav2P+GhOetx94wlC4DC2LeHoTxWQCVhtykjdpk7Y5Qg7Tbq5fxgIu2Dd0gQI06d0JRJipQ19SLNesWUTR9Mf02gf2pmwVtf+5cfpfAvrtrWeOQ+T3V4cUpmrxWRqza9EP53cxd91mydsx7uSrNSaJmM5gzTcjzdn9F4zXf7IVm7HbaOGbJ5ygBgautHL4Ilnf2T6KGzugQ8fFPtVeR+TJoPXhtbzLZm/ToCSzFPQUDK5QU0g0kqxvq/G6e2k1SVbeakpZE174tnamwVJlo0Lo2gahTrhdj7Hvua7Mn3tlLrkosmLFZ750ZWjSSK8eATYAFN2zWTwrmHbJZcLzZymkObMgLA6msyyIgp6X66GQw25kb7deBn0+O3/X4k7KNHe54cOrwgkRa8/GmKwDbgd8uqwT8GtxZDsNd1gbyFF8dZ9ds8DrmbvuwmxD9FmzjxvHa+9ts43Hxzv43IDOqVP4DK0gNKjLZ/pTtVLD/snCHr63y1JBtv5K6qsejfwFQSwECFAMUAAAACAD2s0Vd9unEPWQAAABsAAAAGQAAAAAAAAAAAAAApIEAAAAAcHJvY2VzYW1pZW50by9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAK0RV3HohLNdgcAAEQSAAAiAAAAAAAAAAAAAACkgZsAAABwcm9jZXNhbWllbnRvL2dlbmVyYXJfaGlzdG9yaWFsLnB5UEsBAhQDFAAAAAgA6LNFXT78vHy0CQAA6BwAABgAAAAAAAAAAAAAAKSBUQgAAHByb2Nlc2FtaWVudG8vbW90b3Jlcy5weVBLAQIUAxQAAAAIANS0RV1wjs4UuAYAAO0PAAAWAAAAAAAAAAAAAACkgTsSAABwcm9jZXNhbWllbnRvL21lZGlyLnB5UEsBAhQDFAAAAAgASLRFXav2QrnlBAAAHgsAACAAAAAAAAAAAAAAAKSBJxkAAHByb2Nlc2FtaWVudG8vcGFycXVldF9pbnRlcm5vLnB5UEsFBgAAAAAFAAUAbwEAAEoeAAAAAA=="))).extractall(".")
print("Paquete procesamiento instalado")

In [ ]:
import inspect, os, time
import duckdb, pandas as pd, polars as pl, pyarrow as pa
from IPython.display import Code, display
from procesamiento import generar_historial, motores, parquet_interno
from procesamiento.medir import comparar

def ver(funcion):
    display(Code(inspect.getsource(funcion), language="python"))

print("pandas", pd.__version__, "| Polars", pl.__version__, "| DuckDB", duckdb.__version__, "| CPU:", os.cpu_count())
!free -h | head -2

## Parte 1 · El historial de ventas

Tarda alrededor de un minuto. Genera los mismos 10 millones de filas en tres formatos: un CSV, un Parquet
y un Parquet particionado por año y mes (además de una copia ordenada por fecha que usaremos en la parte 4).

In [ ]:
CARPETA = "datos_historial"
print(generar_historial.generar(CARPETA, filas=10_000_000))
!du -sh datos_historial/*

In [ ]:
pl.read_parquet(f"{CARPETA}/lineas.parquet", n_rows=5)

El mismo contenido: **709 MB en CSV contra unos 170 MB en Parquet**. Parquet comprime por columna y guarda los tipos.

## Parte 2 · Una pregunta, cinco motores

Primero mira cómo se escribe la misma consulta en cada motor:

In [ ]:
ver(motores._pandas_consulta)

In [ ]:
ver(motores._polars_consulta)
print(motores.SQL)

Ahora la carrera. Cada motor corre en **su propio proceso** para medir tiempo y **memoria pico** sin que uno contamine al otro
(ábrelo con `ver(medir)` si tienes curiosidad). Tarda alrededor de un minuto.

In [ ]:
tabla = comparar(CARPETA, motores.MOTORES)
tabla

Lee la tabla con calma:

- **`mismo_resultado`** debe ser `True` en todos. Un motor rápido que da otra respuesta no sirve.
- **pandas sobre CSV** es el caso más común en la práctica, y el más caro: lee las 10 columnas como texto, interpreta fechas y luego filtra.
- **pandas sobre Parquet** ya mejora mucho solo por cambiar el formato y leer las columnas necesarias.
- **Polars diferido y DuckDB** no leen lo que no necesitan: filtran y proyectan **dentro** de la lectura.

## Parte 3 · El plan de Polars

En modo diferido, Polars primero arma un plan y lo optimiza. Compara el plan **como lo escribimos** contra el **optimizado**:

In [ ]:
plan = motores.plan_polars_lazy(CARPETA)
print(plan.explain(optimized=False)[-900:])

In [ ]:
print(plan.explain()[-1300:])

Busca dentro de `Parquet SCAN [lineas.parquet]`:

- **`PROJECT 8/10 COLUMNS`**: *projection pushdown*, solo lee las columnas que usa la consulta.
- **`SELECTION: ...`**: *predicate pushdown*, el filtro de 2025 y canal web se aplica **mientras lee**, no después.

DuckDB hace lo mismo con su propio optimizador:

In [ ]:
d = CARPETA
print(duckdb.sql("EXPLAIN " + motores.SQL.format(tc=17.0, d=d, lineas=f"read_parquet('{d}/lineas.parquet')")).fetchall()[0][1][:1800])

## Parte 4 · Parquet por dentro

Un archivo Parquet se divide en **row groups** (bloques de filas). En el pie del archivo se guardan el mínimo y el máximo
de cada columna en cada row group. Si un filtro cae fuera de ese rango, el motor **ni siquiera lee** ese bloque.

In [ ]:
print("Sin ordenar:")
display(parquet_interno.row_groups(f"{CARPETA}/lineas.parquet").head(4))
print("Ordenado por fecha:")
display(parquet_interno.row_groups(f"{CARPETA}/lineas_ordenado.parquet").head(4))

¿Ves la diferencia? Sin ordenar, **cada** row group va de enero de 2023 a diciembre de 2025: ninguno se puede saltar. Ahora midamos una consulta de diciembre de 2025:

In [ ]:
parquet_interno.comparar_saltos(CARPETA)

Las tres dan el mismo resultado, pero el archivo ordenado lee **1 de 20** row groups y el particionado solo abre la carpeta `mes=12`.
Ordenar y particionar por la columna con la que más filtras es de las optimizaciones más baratas que existen.

## Parte 5 · Más grande que la memoria

Imagina que el CSV no cabe en RAM. Comparamos cuatro estrategias, otra vez cada una en su proceso:

- **pandas completo**: la referencia (carga todo).
- **pandas por partes**: `chunksize`, agregados parciales que se combinan al final.
- **Polars streaming**: el motor procesa el CSV por lotes.
- **DuckDB con 300 MB**: le ponemos un límite de memoria artificialmente bajo; si no le alcanza, usa disco.

In [ ]:
comparar(CARPETA, motores.FUERA_DE_MEMORIA)

In [ ]:
ver(motores.pandas_chunks_csv)

## Parte 6 · Los tipos de datos importan

La misma columna `canal` (solo tres valores distintos) con tres representaciones distintas en pandas:

In [ ]:
canal = pl.read_parquet(f"{CARPETA}/lineas.parquet", columns=["canal"]).to_series().to_pandas()
variantes = {
    "object (texto de Python)": canal.astype(object),
    "string de Arrow": canal.astype("string[pyarrow]"),
    "category": canal.astype("category"),
}
pd.DataFrame({"MB": {k: round(v.memory_usage(deep=True) / 2**20, 1) for k, v in variantes.items()}})

Una columna de texto de baja cardinalidad guardada como objetos de Python puede ocupar decenas de veces más que como `category`.
En pandas, revisar los tipos (`df.dtypes`, `df.memory_usage(deep=True)`) es lo primero cuando algo no cabe.

## Parte 7 · Arrow: el idioma común

pandas (con tipos Arrow), Polars y DuckDB entienden el mismo formato columnar en memoria: **Apache Arrow**.
Pasar datos de uno a otro casi no cuesta.

In [ ]:
lineas_pl = pl.read_parquet(f"{CARPETA}/lineas.parquet", columns=["canal", "cantidad", "precio_unitario"])

inicio = time.perf_counter()
por_canal = duckdb.sql("SELECT canal, SUM(cantidad * precio_unitario) AS venta FROM lineas_pl GROUP BY canal").pl()
print(f"DuckDB consultó un DataFrame de Polars de {len(lineas_pl):,} filas en {time.perf_counter() - inicio:.2f} s")
por_canal

DuckDB encontró la variable `lineas_pl` por su nombre y la leyó directamente a través de Arrow, sin convertirla.
Con `.pl()`, `.df()` o `.arrow()` el resultado vuelve como Polars, pandas o Arrow.

## Retos

### Reto 1 · Tu propia pregunta
Escribe «top 10 productos por ingreso en CDMX en 2025» en **dos** motores y verifica que coinciden.

In [ ]:
# Pista: en DuckDB
# duckdb.sql(f"SELECT producto_id, SUM(...) AS ingreso FROM read_parquet('{CARPETA}/lineas.parquet') l "
#            f"JOIN read_parquet('{CARPETA}/clientes.parquet') c USING (cliente_id) WHERE ... ORDER BY ingreso DESC LIMIT 10").df()

### Reto 2 · Compresión
¿Cuánto cambian el tamaño y el tiempo de escritura con distintos códecs?

In [ ]:
muestra = pl.read_parquet(f"{CARPETA}/lineas.parquet", n_rows=2_000_000)
filas = []
for codec in ["uncompressed", "snappy", "zstd", "gzip"]:
    inicio = time.perf_counter()
    muestra.write_parquet(f"prueba_{codec}.parquet", compression=codec)
    filas.append({"códec": codec, "MB": round(os.path.getsize(f"prueba_{codec}.parquet") / 2**20, 1),
                  "escritura_s": round(time.perf_counter() - inicio, 2)})
    os.remove(f"prueba_{codec}.parquet")
pd.DataFrame(filas).set_index("códec")

### Reto 3 · El problema de los archivos pequeños
El mismo millón de filas en 1 archivo o en 2 000 archivos. ¿Cuánto tarda en leerlo DuckDB?

In [ ]:
import shutil
millon = pl.read_parquet(f"{CARPETA}/lineas.parquet", n_rows=1_000_000)
shutil.rmtree("pequenos", ignore_errors=True); os.makedirs("pequenos")
millon.write_parquet("uno.parquet")
for i, parte in enumerate(millon.iter_slices(500)):
    parte.write_parquet(f"pequenos/parte_{i:05d}.parquet")
for fuente in ["uno.parquet", "pequenos/*.parquet"]:
    inicio = time.perf_counter()
    n = duckdb.execute(f"SELECT COUNT(*), SUM(cantidad) FROM read_parquet('{fuente}')").fetchone()
    print(f"{fuente:22s} {time.perf_counter() - inicio:.3f} s  {n}")

Cada archivo tiene su propio encabezado, metadatos y una apertura en disco (o una petición HTTP si está en Cloud Storage o S3).
Miles de archivos pequeños son lentos y caros: por eso se compactan.

## Cierre

| Hoy viste | Principio |
|---|---|
| CSV contra Parquet | El formato cambia más que el motor |
| Projection y predicate pushdown | No leer lo que no necesitas |
| Row groups ordenados y particiones | Saltar datos es más rápido que procesarlos |
| Streaming y fuera de memoria | Que no quepa en RAM no obliga a usar un clúster |
| Tipos de datos | Revisar `dtypes` antes de comprar más memoria |
| Arrow | Elegir la herramienta por paso, sin pagar conversiones |

**Tarea:** la parte 1.4 del **kit local** (`kit_pipeline_1_4.zip`): corre `medir` en tu máquina y agrega tu pregunta a `motores.py`.

**Próxima sesión (2.1):** pipelines ETL en Python, con transformaciones modulares, deduplicación, SCD y pruebas.

Para liberar espacio en Colab: `!rm -rf datos_historial`